# Yellow Connections Generator

This notebook should generate different combinations of sets of words and corresponding labels that can be used as teh yellow category part of the Connections puzzle. The final pipeline creates JSON objects that can represent these puzzle parts that will be used in the next part of our puzzle generation pipeline.


In [ ]:
# Colab setup

!pip -q install nltk sentence-transformers wordfreq scikit-learn pandas numpy tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 2.8 MB/s eta 0:00:00


In [ ]:
import itertools
import json
import math
import random
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import nltk
from nltk.corpus import wordnet as wn
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import cosine_similarity
from wordfreq import zipf_frequency


def ensure_nltk_resource(resource_name: str, path_name: str) -> None:
    try:
        nltk.data.find(path_name)
    except LookupError:
        nltk.download(resource_name)


ensure_nltk_resource("wordnet", "corpora/wordnet")
ensure_nltk_resource("omw-1.4", "corpora/omw-1.4")


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
# ---------------------------
# Configuration
# ---------------------------

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

MODEL_NAME = "all-MiniLM-L6-v2"

GROUP_SIZE = 4
TARGET_OUTPUTS = 2_500

MIN_WORD_LEN = 3
MAX_WORD_LEN = 12
MIN_WORD_FREQ = 3.6

MIN_CATEGORY_SIZE = 6
MAX_CATEGORY_SIZE = 30

MAX_GROUPS_PER_CATEGORY = 40
STREAM_BATCH_CATEGORIES = 1200
STREAM_MAX_ROUNDS = 60

MIN_YELLOW_SCORE = 0.44
MAX_INTRA_GROUP_JACCARD = 0.50

MIN_CLUSTER_SIZE = 6
MAX_CLUSTER_SIZE = 18

MAX_VOCAB_SIZE = 12_000
EMBED_BATCH_SIZE = 256
KMEANS_BATCH_SIZE = 1024
KMEANS_RANDOM_STATE = RANDOM_SEED

OUTPUT_CSV = "yellow_candidates.csv"
OUTPUT_JSON = "yellow_candidates.json"

model = SentenceTransformer(MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# ---------------------------
# Some word helper functions
# ---------------------------

def normalize_word(word: str) -> str:
    word = word.strip().lower().replace("_", " ")
    word = re.sub(r"\s+", " ", word)
    return word

def singularize_label(label: str) -> str:
    label = normalize_word(label)
    if label.endswith("ies") and len(label) > 4:
        return label[:-3] + "y"
    if label.endswith("ses") and len(label) > 4:
        return label[:-2]
    if label.endswith("s") and not label.endswith("ss") and len(label) > 3:
        return label[:-1]
    return label

def is_clean_word(word: str) -> bool:
    if not word:
        return False
    if " " in word or "-" in word or "_" in word:
        return False
    if not word.isalpha():
        return False
    if len(word) < MIN_WORD_LEN or len(word) > MAX_WORD_LEN:
        return False
    return True

def word_commonness(word: str) -> float:
    return float(zipf_frequency(word.lower(), "en"))

def is_good_candidate_word(word: str) -> bool:
    return is_clean_word(word) and word_commonness(word) >= MIN_WORD_FREQ

def jaccard_similarity(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

def pairwise_cosine_stats(vectors: np.ndarray):
    sims = cosine_similarity(vectors)
    vals = []
    n = sims.shape[0]
    for i in range(n):
        for j in range(i + 1, n):
            vals.append(float(sims[i, j]))
    return {
        "mean_sim": float(np.mean(vals)),
        "min_sim": float(np.min(vals)),
        "max_sim": float(np.max(vals)),
        "std_sim": float(np.std(vals)),
    }


PRETTY_LABEL_REWRITES = {
    "fruit": "types of fruit",
    "vegetable": "types of vegetables",
    "tool": "common tools",
    "garment": "items of clothing",
    "vehicle": "types of vehicles",
    "bird": "types of birds",
    "mammal": "types of mammals",
    "instrument": "musical instruments",
    "furniture": "pieces of furniture",
    "food": "types of food",
    "emotion": "human emotions",
    "weather": "weather-related words",
    "office supply": "common office supplies",
    "kitchenware": "things found in a kitchen",
    "school subject": "school subjects",
    "color": "basic colors",
    "sport": "types of sports",
    "tree": "types of trees",
    "flower": "types of flowers",
    "fish": "types of fish",
    "insect": "types of insects",
}

MEMBER_HINT_LABELS = {
    frozenset({"hammer", "wrench", "pliers", "drill", "saw", "screwdriver"}): "common tools",
    frozenset({"apple", "pear", "peach", "plum", "grape", "melon", "mango", "lemon"}): "types of fruit",
    frozenset({"shirt", "pants", "dress", "sock", "hat", "skirt", "jacket"}): "items of clothing",
    frozenset({"rain", "snow", "fog", "storm", "wind", "hail", "cloud"}): "weather-related words",
    frozenset({"piano", "guitar", "drum", "violin", "flute", "trumpet"}): "musical instruments",
    frozenset({"desk", "chair", "couch", "table", "stool", "shelf"}): "pieces of furniture",
    frozenset({"red", "blue", "green", "yellow", "black", "white", "pink", "brown"}): "basic colors",
}

BAD_ABSTRACT_LABELS = {
    "entity", "object", "artifact", "whole", "unit", "item", "matter",
    "physical entity", "abstraction", "causal agent", "relation"
}

def prettify_label(label: str, members=None, source=None) -> str:
    cleaned = normalize_word(label)
    singular = singularize_label(cleaned)
    member_set = set(members or [])

    if cleaned in PRETTY_LABEL_REWRITES:
        return PRETTY_LABEL_REWRITES[cleaned]
    if singular in PRETTY_LABEL_REWRITES:
        return PRETTY_LABEL_REWRITES[singular]

    for hint_words, pretty in MEMBER_HINT_LABELS.items():
        overlap = len(member_set & set(hint_words))
        if overlap >= min(3, len(hint_words)):
            return pretty

    if singular in BAD_ABSTRACT_LABELS:
        if member_set:
            examples = ", ".join(sorted(list(member_set))[:3])
            return f"words related to {examples}"
        return "related words"

    if source == "cluster":
        if member_set:
            examples = ", ".join(sorted(list(member_set))[:3])
            return f"words related to {examples}"
        return "semantically related words"

    if singular.endswith("ware"):
        base = singular[:-4].strip()
        if base:
            return f"things found in a {base}"
    return f"types of {singular}"


In [ ]:
# ---------------------------
# Build candidate vocabulary
# ---------------------------

def collect_candidate_words(max_vocab_size=MAX_VOCAB_SIZE):
    words = set()

    for syn in wn.all_synsets("n"):
        for lemma in syn.lemmas():
            w = normalize_word(lemma.name())
            if is_good_candidate_word(w):
                words.add(w)

    words = list(words)
    words.sort(key=lambda w: (-word_commonness(w), w))

    if max_vocab_size is not None:
        words = words[:max_vocab_size]

    return words

all_words = collect_candidate_words()
print("Candidate words:", len(all_words))
print(all_words[:40])


Candidate words: 6638
['are', 'have', 'one', 'can', 'will', 'like', 'out', 'more', 'who', 'there', 'time', 'get', 'people', 'now', 'good', 'first', 'know', 'see', 'two', 'make', 'over', 'think', 'then', 'back', 'want', 'well', 'way', 'much', 'even', 'may', 'here', 'need', 'right', 'work', 'year', 'years', 'being', 'day', 'going', 'why']


In [ ]:
# ---------------------------
# Embeddings
# ---------------------------

def embed_words(words, batch_size=EMBED_BATCH_SIZE):
    word_to_vec = {}
    for i in tqdm(range(0, len(words), batch_size), desc="Embedding words"):
        batch = words[i:i + batch_size]
        vecs = model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
        for w, v in zip(batch, vecs):
            word_to_vec[w] = np.asarray(v, dtype=np.float32)
    return word_to_vec

word_to_vec = embed_words(all_words)
print("Embedded words:", len(word_to_vec))


Embedding words:   0%|          | 0/26 [00:00<?, ?it/s]

Embedded words: 6638


In [ ]:
# ---------------------------
# Category sources
# ---------------------------

def build_wordnet_category_bank(
    min_category_size=MIN_CATEGORY_SIZE,
    max_category_size=MAX_CATEGORY_SIZE
):
    category_bank = []
    seen = set()

    for syn in tqdm(list(wn.all_synsets("n")), desc="WordNet categories"):
        hypos = syn.hyponyms()
        if not hypos or not syn.lemmas():
            continue

        members = set()
        for h in hypos:
            for lemma in h.lemmas():
                w = normalize_word(lemma.name())
                if w in word_to_vec and is_good_candidate_word(w):
                    members.add(w)

        members = sorted(members)
        if not (min_category_size <= len(members) <= max_category_size):
            continue

        raw_label = normalize_word(syn.lemmas()[0].name())
        if singularize_label(raw_label) in BAD_ABSTRACT_LABELS:
            continue

        sig = (raw_label, tuple(members))
        if sig in seen:
            continue
        seen.add(sig)

        category_bank.append({
            "source": "wordnet",
            "synset": syn.name(),
            "label": raw_label,
            "pretty_label": prettify_label(raw_label, members=members, source="wordnet"),
            "members": members,
            "size": len(members),
        })

    return category_bank


template_categories = [
    {"source": "template", "label": "fruit", "members": ["apple", "pear", "peach", "plum", "grape", "melon", "mango", "lemon"]},
    {"source": "template", "label": "vegetable", "members": ["carrot", "onion", "pepper", "celery", "radish", "turnip", "cabbage", "spinach"]},
    {"source": "template", "label": "tool", "members": ["hammer", "wrench", "pliers", "drill", "saw", "chisel", "level", "sander"]},
    {"source": "template", "label": "garment", "members": ["shirt", "pants", "dress", "jacket", "skirt", "sock", "hat", "scarf"]},
    {"source": "template", "label": "weather", "members": ["rain", "snow", "wind", "storm", "cloud", "fog", "hail", "sun"]},
    {"source": "template", "label": "school subject", "members": ["math", "science", "history", "english", "music", "art", "physics", "drama"]},
    {"source": "template", "label": "color", "members": ["red", "blue", "green", "yellow", "black", "white", "pink", "brown"]},
    {"source": "template", "label": "instrument", "members": ["piano", "guitar", "drum", "flute", "violin", "trumpet", "cello", "harp"]},
    {"source": "template", "label": "furniture", "members": ["chair", "table", "desk", "couch", "shelf", "stool", "bench", "dresser"]},
]

def sanitize_category_members(category, min_size=GROUP_SIZE):
    cat = dict(category)
    cleaned_members = []
    dropped_members = []

    for member in category["members"]:
        w = normalize_word(member)
        if w in word_to_vec and is_good_candidate_word(w):
            cleaned_members.append(w)
        else:
            dropped_members.append(w)

    cat["members"] = sorted(set(cleaned_members))
    cat["size"] = len(cat["members"])
    cat["pretty_label"] = prettify_label(cat["label"], members=cat["members"], source=cat.get("source"))

    return cat, dropped_members

sanitized_template_categories = []
template_drop_report = {}

for cat in template_categories:
    clean_cat, dropped = sanitize_category_members(cat)
    if dropped:
        template_drop_report[clean_cat["label"]] = dropped
    if clean_cat["size"] >= GROUP_SIZE:
        sanitized_template_categories.append(clean_cat)

template_categories = sanitized_template_categories

if template_drop_report:
    print("Dropped template words missing from the embedding vocabulary:")
    for label, dropped in template_drop_report.items():
        print(f"  {label}: {', '.join(dropped)}")

wordnet_bank = build_wordnet_category_bank()
print("WordNet categories:", len(wordnet_bank))
print("Template categories:", len(template_categories))


Dropped template words missing from the embedding vocabulary:
  fruit: pear, plum, melon, mango
  vegetable: celery, radish, turnip, cabbage, spinach
  tool: wrench, pliers, chisel, sander
  instrument: flute, trumpet, cello, harp
  furniture: stool, dresser


WordNet categories:   0%|          | 0/82115 [00:00<?, ?it/s]

WordNet categories: 707
Template categories: 8


In [ ]:
# ---------------------------
# Embedding-based cluster bank
# ---------------------------

def build_embedding_cluster_bank(words, word_to_vec):
    vocab = [w for w in words if w in word_to_vec]
    X = np.array([word_to_vec[w] for w in vocab], dtype=np.float32)

    # MiniBatchKMeans is much more scalable than agglomerative clustering here.
    n_clusters = max(200, min(len(vocab) // 8, 1500))
    if n_clusters >= len(vocab):
        n_clusters = max(2, len(vocab) // 2)

    clustering = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=KMEANS_RANDOM_STATE,
        batch_size=KMEANS_BATCH_SIZE,
        n_init=10,
    )
    labels = clustering.fit_predict(X)

    clusters = defaultdict(list)
    for w, lab in zip(vocab, labels):
        clusters[int(lab)].append(w)

    cluster_bank = []
    for lab, members in clusters.items():
        members = sorted(set(members))
        if not (MIN_CLUSTER_SIZE <= len(members) <= MAX_CLUSTER_SIZE):
            continue

        vecs = np.array([word_to_vec[w] for w in members], dtype=np.float32)
        center = vecs.mean(axis=0)
        sims = vecs @ center
        raw_label = members[int(np.argmax(sims))]

        cluster_bank.append({
            "source": "cluster",
            "synset": f"cluster_{lab}",
            "label": raw_label,
            "pretty_label": prettify_label(raw_label, members=members, source="cluster"),
            "members": members,
            "size": len(members),
        })

    return cluster_bank

cluster_bank = build_embedding_cluster_bank(all_words, word_to_vec)
print("Cluster categories:", len(cluster_bank))


Cluster categories: 166


In [ ]:
# ---------------------------
# Merge + dedupe category bank
# ---------------------------

def canonical_category_signature(cat):
    return (cat["label"], tuple(sorted(set(cat["members"]))))

full_category_bank = []
seen = set()

for cat in (wordnet_bank + template_categories + cluster_bank):
    members = sorted(set(cat["members"]))
    if len(members) < GROUP_SIZE:
        continue
    cat = dict(cat)
    cat["members"] = members
    cat["size"] = len(members)
    cat["pretty_label"] = prettify_label(cat["label"], members=members, source=cat.get("source"))

    sig = canonical_category_signature(cat)
    if sig in seen:
        continue
    seen.add(sig)
    full_category_bank.append(cat)

print("Total merged categories:", len(full_category_bank))
pd.DataFrame(full_category_bank)[["source", "label", "pretty_label", "size"]].head(20)


Total merged categories: 881


,source,label,pretty_label,size
0,wordnet,thing,types of thing,13
1,wordnet,organism,types of organism,25
2,wordnet,animal,types of animal,17
3,wordnet,natural object,types of natural object,13
4,wordnet,substance,types of substance,11
5,wordnet,substance,types of substance,6
6,wordnet,food,types of food,9
7,wordnet,cognition,types of cognition,18
8,wordnet,motivation,types of motivation,6
9,wordnet,attribute,types of attribute,17


In [ ]:
# ---------------------------
# Scoring + sampling
# ---------------------------

def sample_groups_from_category(category, max_groups=MAX_GROUPS_PER_CATEGORY):
    members = category["members"]
    if len(members) < GROUP_SIZE:
        return []

    if len(members) <= 10:
        combos = list(itertools.combinations(members, GROUP_SIZE))
        random.shuffle(combos)
        return combos[:max_groups]

    combos = set()
    attempts = 0
    max_attempts = max_groups * 25

    while len(combos) < max_groups and attempts < max_attempts:
        attempts += 1
        group = tuple(sorted(random.sample(members, GROUP_SIZE)))
        combos.add(group)

    return list(combos)


def score_group_yellow(group, label):
    normalized_group = tuple(normalize_word(w) for w in group)
    missing_words = [w for w in normalized_group if w not in word_to_vec]

    if missing_words:
        return {
            "valid": False,
            "missing_words": missing_words,
            "score": float("-inf"),
            "mean_sim": float("nan"),
            "min_sim": float("nan"),
            "std_sim": float("nan"),
            "avg_freq": float("nan"),
            "min_freq": float("nan"),
        }

    freqs = [word_commonness(w) for w in normalized_group]
    vecs = np.array([word_to_vec[w] for w in normalized_group], dtype=np.float32)
    sim_stats = pairwise_cosine_stats(vecs)

    avg_freq = float(np.mean(freqs))
    min_freq = float(np.min(freqs))
    coherence = sim_stats["mean_sim"]
    min_coherence = sim_stats["min_sim"]
    spread_penalty = sim_stats["std_sim"]

    score = (
        0.42 * coherence
        + 0.18 * min_coherence
        + 0.28 * (avg_freq / 8.0)
        + 0.12 * (min_freq / 8.0)
        - 0.10 * spread_penalty
    )

    return {
        "valid": True,
        "missing_words": [],
        "score": float(score),
        "mean_sim": coherence,
        "min_sim": min_coherence,
        "std_sim": spread_penalty,
        "avg_freq": avg_freq,
        "min_freq": min_freq,
    }


In [ ]:
# ---------------------------
# Novelty memory
# ---------------------------

class NoveltyMemory:
    def __init__(self):
        self.groups = []
        self.labels = []
        self.used_words_by_label = defaultdict(set)

    def is_novel(self, label, group):
        g = set(group)

        for old_label, old_group in zip(self.labels, self.groups):
            old_group = set(old_group)
            group_sim = jaccard_similarity(g, old_group)

            if label == old_label and group_sim >= 0.25:
                return False
            if group_sim >= MAX_INTRA_GROUP_JACCARD:
                return False

        if len(g & self.used_words_by_label[label]) >= 3:
            return False

        return True

    def add(self, label, group):
        self.labels.append(label)
        self.groups.append(tuple(group))
        self.used_words_by_label[label].update(group)


In [ ]:
# ---------------------------
# Stream generator
# ---------------------------

def generate_yellow_stream(
    category_bank,
    target_n=2500,
    batch_categories=STREAM_BATCH_CATEGORIES,
    max_groups_per_category=MAX_GROUPS_PER_CATEGORY,
    min_score=MIN_YELLOW_SCORE,
    max_rounds=STREAM_MAX_ROUNDS,
):
    memory = NoveltyMemory()
    outputs = []
    rounds = 0
    skipped_missing_words = Counter()

    while len(outputs) < target_n and rounds < max_rounds:
        rounds += 1
        sampled_categories = random.sample(
            category_bank,
            min(batch_categories, len(category_bank))
        )

        for cat in tqdm(sampled_categories, desc=f"Generation round {rounds}", leave=False):
            label = cat["label"]
            pretty_label = cat["pretty_label"]
            groups = sample_groups_from_category(cat, max_groups=max_groups_per_category)

            for group in groups:
                stats = score_group_yellow(group, label)

                if not stats["valid"]:
                    skipped_missing_words.update(stats["missing_words"])
                    continue

                if stats["score"] < min_score:
                    continue

                if not memory.is_novel(label, group):
                    continue

                item = {
                    "label": label,
                    "pretty_label": pretty_label,
                    "group": tuple(group),
                    "group_str": ", ".join(group),
                    "source": cat["source"],
                    "category_size": cat["size"],
                    **{k: v for k, v in stats.items() if k not in {"valid", "missing_words"}},
                }

                memory.add(label, group)
                outputs.append(item)

                if len(outputs) >= target_n:
                    break

            if len(outputs) >= target_n:
                break

        print(f"After round {rounds}: {len(outputs)} outputs")

        if rounds in {8, 16, 24, 32} and len(outputs) < target_n:
            min_score = max(0.39, min_score - 0.01)
            print(f"Relaxed score threshold to {min_score:.3f}")

    if skipped_missing_words:
        print("Skipped out-of-vocabulary words during scoring:")
        print(dict(skipped_missing_words.most_common(20)))

    return outputs

final_outputs = generate_yellow_stream(full_category_bank, target_n=TARGET_OUTPUTS)
print("Generated outputs:", len(final_outputs))


Generation round 1:   0%|          | 0/881 [00:00<?, ?it/s]

After round 1: 1041 outputs


Generation round 2:   0%|          | 0/881 [00:00<?, ?it/s]

After round 2: 1165 outputs


Generation round 3:   0%|          | 0/881 [00:00<?, ?it/s]

...

After round 58: 2183 outputs


Generation round 59:   0%|          | 0/881 [00:00<?, ?it/s]

After round 59: 2187 outputs


Generation round 60:   0%|          | 0/881 [00:00<?, ?it/s]

After round 60: 2187 outputs
Generated outputs: 2187


In [ ]:
# ---------------------------
# Review results
# ---------------------------

df = pd.DataFrame(final_outputs)

if df.empty:
    raise ValueError("No outputs were generated. Try lowering MIN_YELLOW_SCORE or increasing MAX_VOCAB_SIZE.")

df["n_words"] = df["group"].apply(len)

print("Rows:", len(df))
print("Unique raw labels:", df["label"].nunique())
print("Unique pretty labels:", df["pretty_label"].nunique())
print("Unique sources:", df["source"].nunique())
print()
print(df["source"].value_counts())

df.sort_values("score", ascending=False).head(20)[
    ["pretty_label", "label", "group_str", "source", "score", "mean_sim", "avg_freq"]
]


Rows: 2187
Unique raw labels: 742
Unique pretty labels: 758
Unique sources: 3

source
wordnet     1643
cluster      532
template      12
Name: count, dtype: int64


,pretty_label,label,group_str,source,score,mean_sim,avg_freq
345,"words related to friday, monday, saturday",saturday,"monday, saturday, sunday, tuesday",cluster,0.709878,0.805177,4.8050
22,"words related to buy, buyer, sale",selling,"buy, sale, sell, selling",cluster,0.698063,0.792080,5.0250
288,"words related to daughter, family, mama",mother,"mom, mommy, mother, mum",cluster,0.694163,0.810482,4.6650
283,"words related to dialogue, speaking, speech",talking,"speaking, talk, talking, talks",cluster,0.676151,0.767063,5.0575
23,"words related to buy, buyer, sale",selling,"buyer, seller, sellers, selling",cluster,0.667175,0.800304,4.1725
140,types of spot,spot,"eight, five, four, seven",wordnet,0.663421,0.697208,5.2700
955,types of gregorian calendar month,gregorian calendar month,"april, august, february, january",wordnet,0.654745,0.685272,5.1025
139,types of spot,spot,"five, nine, six, ten",wordnet,0.651676,0.699161,5.1650
320,basic colors,chromatic color,"blue, pink, purple, red",wordnet,0.647323,0.706299,4.8225
485,"words related to bush, clinton, hillary",president,"bush, presidency, president, trump",cluster,0.645750,0.741283,4.7575


In [ ]:
# Optional: inspect label diversity
top_labels = (
    df.groupby(["label", "pretty_label"])
      .size()
      .sort_values(ascending=False)
      .reset_index(name="count")
)
top_labels.head(25)


,label,pretty_label,count
0,part,types of part,14
1,region,types of region,13
2,area,types of area,12
3,communication,types of communication,12
4,body,types of body,11
5,point,types of point,11
6,blow,types of blow,11
7,speech act,types of speech act,11
8,position,types of position,11
9,content,types of content,10


In [ ]:
# ---------------------------
# Save outputs
# ---------------------------

df_to_save = df.copy()
df_to_save["group"] = df_to_save["group"].apply(list)

df_to_save.to_csv(OUTPUT_CSV, index=False)

dataset_json = {
    "dataset_name": "yellow_connections_candidates",
    "target_count": int(TARGET_OUTPUTS),
    "actual_count": int(len(df_to_save)),
    "puzzles": [
        {
            "id": int(i),
            "category_color": "yellow",
            "label": row["label"],
            "pretty_label": row["pretty_label"],
            "group": list(row["group"]),
            "group_str": row["group_str"],
            "source": row["source"],
            "category_size": int(row["category_size"]),
            "score": float(row["score"]),
            "mean_sim": float(row["mean_sim"]),
            "min_sim": float(row["min_sim"]),
            "std_sim": float(row["std_sim"]),
            "avg_freq": float(row["avg_freq"]),
            "min_freq": float(row["min_freq"]),
        }
        for i, row in enumerate(df_to_save.to_dict(orient="records"), start=1)
    ],
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(dataset_json, f, indent=2)

print(f"Saved CSV  -> {OUTPUT_CSV}")
print(f"Saved JSON -> {OUTPUT_JSON}")


Saved CSV  -> yellow_candidates_10k.csv
Saved JSON -> yellow_candidates_10k.json


In [ ]:
# ---------------------------
# Utility: display random examples
# ---------------------------

def show_outputs(outputs_df, n=25, seed=42):
    sample_df = outputs_df.sample(min(n, len(outputs_df)), random_state=seed)
    for i, row in enumerate(sample_df.itertuples(index=False), start=1):
        print(f"{i}. CATEGORY: {row.pretty_label}")
        print(f"   RAW LABEL: {row.label}")
        print(f"   WORDS: {row.group_str}")
        print(f"   source={row.source} | score={row.score:.3f} | mean_sim={row.mean_sim:.3f} | avg_freq={row.avg_freq:.2f}")
        print()

show_outputs(df, n=25)


1. CATEGORY: types of proceeding
   RAW LABEL: proceeding
   WORDS: bankruptcy, lawsuit, litigation, suit
   source=wordnet | score=0.437 | mean_sim=0.480 | avg_freq=4.21

2. CATEGORY: types of large indefinite quantity
   RAW LABEL: large indefinite quantity
   WORDS: loads, lot, maximum, plenty
   source=wordnet | score=0.477 | mean_sim=0.442 | avg_freq=4.79

3. CATEGORY: types of substance
   RAW LABEL: substance
   WORDS: food, fuel, poison, vehicle
   source=wordnet | score=0.441 | mean_sim=0.426 | avg_freq=4.74

4. CATEGORY: types of indefinite quantity
   RAW LABEL: indefinite quantity
   WORDS: gain, output, production, step
   source=wordnet | score=0.434 | mean_sim=0.358 | avg_freq=4.88

5. CATEGORY: types of taxonomic group
   RAW LABEL: taxonomic group
   WORDS: form, order, type, variety
   source=wordnet | score=0.443 | mean_sim=0.373 | avg_freq=5.21

6. CATEGORY: types of merchandise
   RAW LABEL: merchandise
   WORDS: feature, generic, inventory, number
   source=wordne

In [ ]:
# ---------------------------
# Quick validation
# ---------------------------

assert "pretty_label" in df.columns
assert df["group"].apply(len).eq(4).all()
assert df["score"].notna().all()

with open(OUTPUT_JSON, "r") as f:
    exported = json.load(f)

assert exported["actual_count"] == len(df)
assert all(len(p["group"]) == 4 for p in exported["puzzles"])

print("Validation passed.")


Validation passed.
